# Triple Interaction Analysis with MICE Imputation on OWID Panel Data

## Overview

This notebook implements a confirmatory test of the triple interaction hypothesis (inequality × education × welfare spending) on democratic resilience using panel data from post-1990 democratizers.

### What this analysis does:
1. **Data Loading**: Loads OWID panel data with V-Dem democracy scores, Gini inequality, education, and social spending
2. **Gini Correction**: Fixes mixed 0-1 and 0-100 scale errors in Gini coefficients
3. **MICE Imputation**: Handles missing data using Multiple Imputation by Chained Equations
4. **Panel Regression**: Runs fixed-effects panel regressions with clustered standard errors
5. **Triple Interaction Test**: Tests whether inequality × education × welfare spending predicts democratic resilience

### Key Output
- Triple interaction coefficient and p-value
- Marginal effects at different levels of education and welfare spending


## Cell 1: Install Dependencies

Install all required packages. Packages pre-installed on Colab are only installed locally to match Colab versions.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru', 'miceforest', 'linearmodels')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'statsmodels==0.14.6')


## Cell 2: Imports

Import all required libraries.


In [ ]:
from loguru import logger
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import json
import sys
import numpy as np
import pandas as pd
import gc
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

import matplotlib.pyplot as plt
%matplotlib inline

print("All imports successful!")


## Cell 3: Data Loading Helper

Loads mini_demo_data.json from GitHub (for Colab) with local fallback (for testing).


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-90d6bf-the-triple-shield-revisited-education-we/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load mini_demo_data.json from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    if os.path.exists("mini_demo_data.json"):
        print("Using local mini_demo_data.json")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Test the loading function
try:
    data = load_data()
    print(f"Data loaded successfully!")
    print(f"Number of examples: {len(data['datasets'][0]['examples'])}")
except Exception as e:
    print(f"Error loading data: {e}")


## Cell 4: Configuration

Define all tunable parameters. Start with ABSOLUTE MINIMUM values for fast demo execution.

**Demo values** (minimize runtime):
- N_IMPUTATIONS = 3 (original: 50)
- N_ITERATIONS = 2 (original: 10)


In [ ]:
# ===========================================================================
# CONFIGURATION - Set to minimum values for demo
# ===========================================================================

# MICE Imputation parameters
N_IMPUTATIONS = 3  # Original: 50
N_ITERATIONS = 2    # Original: 10
RANDOM_STATE = 42    # For reproducibility

print("Configuration set:")
print(f"  N_IMPUTATIONS = {N_IMPUTATIONS}")
print(f"  N_ITERATIONS = {N_ITERATIONS}")


## Cell 5: Data Loading and Verification

Convert the JSON data into a pandas DataFrame with multi-index (Code, Year).


In [ ]:
# ===========================================================================
# DATA LOADING AND VERIFICATION
# ===========================================================================

from dataclasses import dataclass

@dataclass
class RegressionResult:
    """Container for regression results."""
    model_name: str
    coefficients: Dict[str, float]
    std_errors: Dict[str, float]
    p_values: Dict[str, float]
    n_obs: int
    r_squared: float

def load_and_verify_data(data) -> pd.DataFrame:
    """Load the panel dataset from the JSON data structure."""
    logger.info("Loading data from JSON...")
    
    examples = data['datasets'][0]['examples']
    logger.info(f"Loaded {len(examples)} examples from JSON")
    
    rows = []
    for i, example in enumerate(examples):
        try:
            input_dict = json.loads(example['input'])
            country = example.get('metadata_country', f'UNK_{i}')
            year = example.get('metadata_year', 1990 + i)
            vdem_electoral = float(example['output'])
            
            row = {
                'Code': country,
                'Year': int(year),
                'vdem_electoral': vdem_electoral,
                **input_dict
            }
            rows.append(row)
        except Exception as e:
            logger.warning(f"Failed to parse example {i}: {e}")
            continue
    
    df = pd.DataFrame(rows)
    
    numeric_cols = ['vdem_electoral', 'gini', 'education_years', 'social_spending',
                    'gdp_per_capita', 'resource_rents', 'population']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.set_index(['Code', 'Year'])
    
    logger.info(f"DataFrame shape: {df.shape}")
    return df

# Load the data
df = load_and_verify_data(data)
print("\nData loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())


## Cell 6: Fix Gini Scaling Error

The dataset has mixed Gini scales (some 0-1, some 0-100). This cell identifies and fixes the scaling error.


In [ ]:
# ===========================================================================
# FIX GINI SCALING ERROR
# ===========================================================================

def diagnose_gini_scaling(df):
    """Diagnose Gini scaling."""
    gini_col = 'gini'
    if gini_col not in df.columns:
        return {}
    
    gini_data = df[gini_col].dropna()
    if len(gini_data) == 0:
        return {}
    
    diagnostics = {
        'count': len(gini_data),
        'mean': float(gini_data.mean()),
        'values_gt_1': int((gini_data > 1.0).sum()),
    }
    
    if diagnostics['values_gt_1'] > 0:
        diagnostics['needs_fix'] = True
    else:
        diagnostics['needs_fix'] = False
    
    return diagnostics

def fix_gini_scaling(df):
    """Fix Gini scaling by converting values > 1 to 0-1 scale."""
    df = df.copy()
    gini_col = 'gini'
    
    if gini_col not in df.columns:
        return df
    
    mask = df[gini_col] > 1.0
    n_fixed = mask.sum()
    
    if n_fixed > 0:
        print(f"Fixing {n_fixed} Gini values by dividing by 100")
        df.loc[mask, gini_col] = df.loc[mask, gini_col] / 100.0
    
    return df

# Fix Gini scaling
print("Before fix:")
diag = diagnose_gini_scaling(df)
print(f"  Gini > 1.0: {diag.get('values_gt_1', 0)}")

df = fix_gini_scaling(df)

print("\nAfter fix:")
diag = diagnose_gini_scaling(df)
print(f"  Gini mean: {diag.get('mean', 0):.4f}")


## Cell 7: Prepare Data for Imputation

Prepare the DataFrame for MICE imputation.


In [ ]:
# ===========================================================================
# PREPARE DATA FOR IMPUTATION
# ===========================================================================

def prepare_data_for_imputation(df):
    """Prepare data for MICE imputation."""
    df_index = df.reset_index()[['Code', 'Year']]
    df_imp = df.reset_index(drop=True)
    
    cols_for_imputation = ['vdem_electoral', 'gini', 'education_years', 
                          'social_spending', 'gdp_per_capita', 
                          'resource_rents', 'population']
    
    df_imp = df_imp[cols_for_imputation]
    return df_imp, df_index

df_imp, df_index = prepare_data_for_imputation(df)
print(f"Data prepared for imputation: {df_imp.shape}")
print(f"Missing values:\n{df_imp.isnull().sum()}")


## Cell 8: Run MICE Imputation

Run Multiple Imputation by Chained Equations (MICE) to handle missing data.
Note: Using N_IMPUTATIONS=3 and N_ITERATIONS=2 for fast demo.


In [ ]:
# ===========================================================================
# RUN MICE IMPUTATION
# ===========================================================================

from dataclasses import dataclass

@dataclass
class ImputationResult:
    """Container for MICE imputation results."""
    imputed_datasets: list
    n_imputations: int

def run_mice_imputation(df, n_imputations=3, n_iterations=2, random_state=42):
    """Run MICE multiple imputation."""
    import miceforest as mf
    
    print(f"Starting MICE with {n_imputations} imputations...")
    
    kernel = mf.ImputationKernel(
        df,
        num_datasets=n_imputations,
        save_all_iterations_data=False,
        random_state=random_state,
        copy_data=True
    )
    
    kernel.mice(iterations=n_iterations, verbose=False)
    
    imputed_datasets = []
    for i in range(n_imputations):
        imp_df = kernel.complete_data(i)
        imputed_datasets.append(imp_df)
    
    print(f"MICE complete. Generated {len(imputed_datasets)} datasets.")
    
    return ImputationResult(
        imputed_datasets=imputed_datasets,
        n_imputations=n_imputations
    )

# Run MICE
imp_result = run_mice_imputation(df_imp, n_imputations=N_IMPUTATIONS, n_iterations=N_ITERATIONS, random_state=RANDOM_STATE)
print("\nImputation complete!")


## Cell 9: Create Interaction Terms

Create standardized interaction terms for the triple interaction model.
Standardize variables and create: gini × education, gini × social, triple interaction.


In [ ]:
# ===========================================================================
# CREATE INTERACTION TERMS
# ===========================================================================

def add_index_info(df, df_index):
    """Add Code and Year columns back and set MultiIndex."""
    df = df.copy()
    df['Code'] = df_index['Code'].values
    df['Year'] = df_index['Year'].values
    return df.set_index(['Code', 'Year'])

def create_interaction_terms(df):
    """Create standardized interaction terms."""
    df = df.copy()
    
    # Standardize variables
    for var in ['gini', 'education_years', 'social_spending']:
        if var in df.columns:
            mean = df[var].mean()
            std = df[var].std()
            if std > 0:
                df[f'{var}_z'] = (df[var] - mean) / std
            else:
                df[f'{var}_z'] = 0
    
    # Create interactions
    df['gini_x_edu'] = df['gini_z'] * df['education_years_z']
    df['gini_x_soc'] = df['gini_z'] * df['social_spending_z']
    df['triple_int'] = df['gini_z'] * df['education_years_z'] * df['social_spending_z']
    
    return df

# Add index and create interactions
imp_df_with_index = add_index_info(imp_result.imputed_datasets[0], df_index)
df_with_interactions = create_interaction_terms(imp_df_with_index)

print("Interaction terms created!")
print(f"New columns: {[col for col in df_with_interactions.columns if '_z' in col or '_x_' in col or 'triple' in col]}")


## Cell 10: Panel Regression (Fixed Effects)

Run fixed-effects panel regression using linearmodels.
Tests the triple interaction hypothesis.


In [ ]:
# ===========================================================================
# PANEL REGRESSION (Fixed Effects)
# ===========================================================================

def run_panel_regression(df, model_spec="triple"):
    """Run fixed-effects panel regression."""
    from linearmodels.panel import PanelOLS
    import statsmodels.api as sm
    
    df_reg = df.copy()
    if not isinstance(df_reg.index, pd.MultiIndex):
        df_reg = df_reg.set_index(['Code', 'Year'])
    
    y = 'vdem_electoral'
    exog_vars = ['gini_z', 'education_years_z', 'social_spending_z',
                 'gini_x_edu', 'gini_x_soc', 'triple_int',
                 'gdp_per_capita', 'resource_rents']
    
    exog = sm.add_constant(df_reg[exog_vars])
    endog = df_reg[y]
    
    try:
        model = PanelOLS(endog, exog, entity_effects=True, time_effects=True, drop_absorbed=True)
        result = model.fit(cov_type='clustered', cluster_entity=True)
        
        return RegressionResult(
            model_name="Triple Interaction Model",
            coefficients=result.params.to_dict(),
            std_errors=result.std_errors.to_dict(),
            p_values=result.pvalues.to_dict(),
            n_obs=int(result.nobs),
            r_squared=float(result.rsquared)
        )
    except Exception as e:
        print(f"Regression failed: {e}")
        return None

# Run regression
result = run_panel_regression(df_with_interactions)

if result:
    print(f"\n{result.model_name}")
    print(f"N = {result.n_obs}")
    print(f"R-squared = {result.r_squared:.4f}")
    print(f"\nTriple interaction coefficient: {result.coefficients.get('triple_int', 0):.4f}")
    print(f"p-value: {result.p_values.get('triple_int', 1):.4f}")
else:
    print("Regression failed!")


## Cell 11: Results Visualization

Display key results in a readable format and create visualizations.


In [ ]:
# ===========================================================================
# VISUALIZATION AND RESULTS
# ===========================================================================

if result:
    # Print summary
    print("="*80)
    print("REGRESSION RESULTS SUMMARY")
    print("="*80)
    
    for var in result.coefficients.keys():
        if var == 'const':
            continue
        coef = result.coefficients.get(var, 0)
        se = result.std_errors.get(var, 0)
        pval = result.p_values.get(var, 1)
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f"{var:25s}: {coef:8.4f} (SE={se:.4f}){sig}")
    
    # Plot coefficients
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    vars_to_plot = [v for v in result.coefficients.keys() if v != 'const']
    coefs = [result.coefficients[v] for v in vars_to_plot]
    colors = ['green' if result.p_values[v] < 0.05 else 'gray' for v in vars_to_plot]
    
    ax.barh(vars_to_plot, coefs, color=colors)
    ax.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Coefficient Value')
    ax.set_title('Regression Coefficients (Green = p < 0.05)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print primary hypothesis test
    print("\n" + "="*80)
    print("PRIMARY HYPOTHESIS TEST: Triple Interaction")
    print("="*80)
    triple_coef = result.coefficients.get('triple_int', 0)
    triple_pval = result.p_values.get('triple_int', 1)
    print(f"Coefficient: {triple_coef:.4f}")
    print(f"p-value: {triple_pval:.4f}")
    if triple_pval < 0.05:
        print("RESULT: Statistically significant (p < 0.05)")
    else:
        print("RESULT: Not statistically significant (p >= 0.05)")
else:
    print("No results to display.")
